# Introduction to LangChain

Calling a large language model (LLM) API is straightforward: send text, receive text. Building a **product** around that API is not. Real assistants retrieve documents, call tools, format structured output, stream tokens, remember conversation history, retry on failure, and log every step for debugging. Each concern adds code — prompt assembly, provider adapters, error handling — that has little to do with your business logic.

**LangChain** is an open-source Python (and JavaScript) framework for composing LLM-powered applications. It provides a shared **`Runnable`** interface, **LCEL** (LangChain Expression Language) for piping steps together, and integrations for OpenAI, Anthropic, Chroma, and dozens of other providers and data stores. Instead of writing bespoke glue for every pipeline, you assemble reusable components.

LangChain does **not** replace the LLM. It sits between your application and provider SDKs, standardizing how prompts, models, parsers, retrievers, and tools connect. Modern LangChain (1.x) is modular: **`langchain-core`** holds primitives; provider packages (`langchain-openai`, `langchain-chroma`) add integrations; the top-level **`langchain`** package adds agents and convenience helpers.

This notebook defines LangChain, explains why it exists, maps the package architecture, previews core concepts you will implement hands-on in later notebooks, compares LangChain to raw SDK code, and verifies your environment. It is the entry point for the **01. LangChain** track — each notebook stands alone but follows a deliberate sequence from runnables through production patterns.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json

import langchain
import langchain_core
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

print("langchain:", langchain.__version__)
print("langchain-core:", langchain_core.__version__)

---

## 1. The Problem LangChain Solves

The first version of almost every GenAI app is a single function:

```python
def ask(question: str) -> str:
    response = openai_client.chat.completions.create(...)
    return response.choices[0].message.content
```

That function grows. You add a system prompt. Then retrieved documents. Then JSON parsing. Then tool calling. Then streaming. Then session memory. Soon one file contains hundreds of lines mixing **prompt engineering**, **HTTP details**, and **domain logic**.

| Symptom | Without a framework |
|---------|---------------------|
| Duplicate prompt strings | Copy-paste across routes |
| Provider lock-in | OpenAI types everywhere |
| No replay of failures | Logs show final answer only |
| Inconsistent async/stream | Reimplemented per endpoint |
| Hard to test | Must mock HTTP, not steps |

LangChain addresses this by treating each step as a **composable unit** with a standard interface. Your product code describes *what* the pipeline does; LangChain handles *how* data flows between steps.

---

## 2. What Is LangChain?

**LangChain** is a library ecosystem for building **LLM applications** through composition. Key ideas:

1. **Everything is a Runnable** — prompts, models, parsers, retrievers share `invoke`, `stream`, `batch`, and async variants.
2. **Pipe to compose** — chain steps with `|` (LCEL): `prompt | model | parser`.
3. **Integrations are pluggable** — swap OpenAI for Anthropic, Chroma for Pinecone, with minimal code change.
4. **Observability is first-class** — callbacks and LangSmith tracing hook into every Runnable.

LangChain is **not**:

| Misconception | Reality |
|---------------|--------|
| A hosted LLM | You bring your own API keys and models |
| A vector database | It integrates with Chroma, Pinecone, etc. |
| Required for RAG | RAG is a pattern; you can build it in plain Python |
| A replacement for your backend | FastAPI, auth, and databases remain yours |

Think of LangChain as **orchestration glue** for generative AI — similar to how SQLAlchemy orchestrates database access without being a database.

---

## 3. Where LangChain Sits in Your Stack

```
┌─────────────────────────────────────────────────────────────┐
│  Your application (FastAPI, CLI, Slack bot, batch job)      │
├─────────────────────────────────────────────────────────────┤
│  LangChain — chains, retrievers, tools, agents, memory      │
├─────────────────────────────────────────────────────────────┤
│  Provider SDKs — OpenAI, Anthropic, Google                  │
├─────────────────────────────────────────────────────────────┤
│  Data layer — vector DB, Redis, PostgreSQL, S3              │
└─────────────────────────────────────────────────────────────┘
```

LangChain never sees end users directly. Your API layer validates requests, enforces auth, and calls LangChain runnables. Results are mapped to your response DTOs before returning to the client.

For **complex cyclic workflows** (agent loops with checkpointing, human-in-the-loop interrupts), LangChain 1.x uses **LangGraph** under the hood for `create_agent`. Explicit graph authoring is a separate topic outside this LangChain track — here we focus on LangChain's Runnable and LCEL surface.

---

## 4. Package Architecture

LangChain split into focused packages so you install only what you need:

| Package | Purpose |
|---------|--------|
| **`langchain-core`** | `Runnable`, messages, prompts, parsers, tools, callbacks — the foundation |
| **`langchain`** | `create_agent`, `init_chat_model`, high-level helpers |
| **`langchain-openai`** | `ChatOpenAI`, `OpenAIEmbeddings` |
| **`langchain-chroma`** | `Chroma` vector store integration |
| **`langchain-text-splitters`** | `RecursiveCharacterTextSplitter` and other splitters |
| **`langchain-community`** | Optional: hundreds of loaders and third-party integrations |

```
                    langchain-core
                          ▲
          ┌───────────────┼───────────────┐
          │               │               │
  langchain-openai  langchain-chroma  langchain-text-splitters
          │               │               │
          └───────────────┼───────────────┘
                          ▲
                      langchain
```

**Rule of thumb:** import from the most specific package. Use `langchain_core.prompts`, not a catch-all `from langchain import ...`, when you know the module.

In [ ]:
# Verify optional integration packages (install from requirements.txt if missing)

packages = {}

for name, import_stmt in {
    "langchain-openai": "from langchain_openai import ChatOpenAI",
    "langchain-chroma": "from langchain_chroma import Chroma",
    "langchain-text-splitters": "from langchain_text_splitters import RecursiveCharacterTextSplitter",
}.items():
    try:
        exec(import_stmt)
        packages[name] = "OK"
    except ImportError as e:
        packages[name] = f"missing ({e.name})"

print(json.dumps(packages, indent=2))

---

## 5. LangChain 1.x — The Modern Stack

LangChain evolved rapidly. If you find tutorials referencing `LLMChain`, `from langchain.llms import OpenAI`, or `AgentExecutor`, those are **legacy patterns**.

LangChain **1.x** promotes:

| Old (avoid in new code) | Modern replacement |
|-------------------------|-------------------|
| `LLMChain` | LCEL: `prompt \| llm \| parser` |
| Completion `OpenAI` LLM | `ChatOpenAI` chat model |
| `create_tool_calling_agent` + `AgentExecutor` | `create_agent` |
| Monolithic `langchain` imports | `langchain_core`, `langchain_openai`, etc. |

This **01. LangChain** track teaches 1.x idioms only. Pin versions in production (`langchain>=1.0`) and read migration notes when upgrading — breaking changes are documented on [docs.langchain.com](https://docs.langchain.com).

---

## 6. The Runnable — LangChain's Universal Interface

Every LangChain component implements **`Runnable`**: a unit of work with a standard execution API.

| Method | Description |
|--------|-------------|
| `invoke(input)` | Single synchronous call |
| `batch([inputs])` | Process many inputs |
| `stream(input)` | Yield incremental output chunks |
| `ainvoke` / `astream` / `abatch` | Async equivalents |

Runnables compose with the pipe operator **`|`**:

$$\text{chain.invoke}(x) = \text{parser}(\text{llm}(\text{prompt}(x)))$$

Because prompts, models, and parsers are all Runnables, the same composition syntax works everywhere. **02. Runnable Protocol and LCEL** covers this in full depth.

In [ ]:
from langchain_core.runnables import RunnableLambda


def to_upper(text: str) -> str:
    return text.upper()


def add_prefix(text: str) -> str:
    return f"RESULT: {text}"


# Three Runnables composed with LCEL — no LLM required
demo_chain = RunnableLambda(to_upper) | RunnableLambda(add_prefix)

print("invoke:", demo_chain.invoke("hello langchain"))
print("batch:", demo_chain.batch(["alpha", "beta", "gamma"]))
print("runnable type:", type(demo_chain).__name__)

---

## 7. LCEL — LangChain Expression Language

**LCEL** is the syntax and convention for composing Runnables. The canonical LLM chain looks like:

```python
chain = prompt | llm | output_parser
answer = chain.invoke({"question": "What is JWT?"})
```

Benefits over imperative code:

| Benefit | Why it matters |
|---------|----------------|
| **Declarative** | Data flow is visible at a glance |
| **Streaming** | `chain.stream()` works without extra wiring |
| **Parallelism** | `batch()` parallelizes where providers allow |
| **Callbacks** | Tracing attaches to the whole chain |
| **Type hints** | Input/output schemas propagate (with parsers) |

Advanced composition — passing dicts, fan-out, conditional branches — uses helpers like `RunnablePassthrough`, `RunnableParallel`, and `RunnableBranch`. Those are covered in **06. LCEL Composition Patterns**.

---

## 8. Core Building Blocks — Map of the Track

LangChain applications combine these primitives. Each gets a dedicated notebook in this series:

| Building block | Role | Notebook |
|----------------|------|----------|
| **Runnable / LCEL** | Compose any pipeline | **02** |
| **Chat model** | Call OpenAI, Anthropic, etc. | **03** |
| **Prompt template** | Variable messages, few-shot | **04** |
| **Output parser** | Text → str, JSON, Pydantic | **05** |
| **Composition helpers** | Parallel, branch, passthrough | **06** |
| **Streaming / async** | Token streams, `ainvoke` | **07** |
| **Document + splitter** | Chunk text for retrieval | **08** |
| **Embeddings + vector store** | Index and search | **09** |
| **Retriever** | `query → list[Document]` | **10** |
| **RAG chain** | Retrieve → prompt → generate | **11** |
| **Tool** | Function the model can call | **12** |
| **Agent** | Multi-step tool loop | **13** |
| **Memory** | Conversation history | **14** |
| **Callbacks / cache** | Tracing, cost control | **15** |
| **Fallbacks / config** | Resilience, runtime params | **16** |
| **Production wrapper** | Service layer, testing | **17** |

You do not need every block for every app. A summarizer uses `prompt | llm | parser`. A doc bot adds retriever and RAG. An action bot adds tools and agents.

---

## 9. From Single Call to Full Application

Complexity grows in predictable stages:

```
Stage 1:  prompt | llm | StrOutputParser          → chatbot
Stage 2:  + ChatPromptTemplate with variables     → reusable prompts
Stage 3:  + retriever | format_docs in chain      → RAG Q&A
Stage 4:  + @tool + bind_tools or create_agent    → live data / actions
Stage 5:  + RunnableWithMessageHistory            → multi-turn
Stage 6:  + callbacks, cache, fallbacks           → production hardening
```

LangChain supports every stage with the **same Runnable interface**. You can ship Stage 1 in an afternoon and add retrieval later without rewriting the model layer.

---

## 10. Messages — The Chat Model Contract

Modern LangChain uses **chat models** that accept a **list of messages**, not a single raw string.

| Message type | Role | Typical use |
|--------------|------|-------------|
| `SystemMessage` | system | Persona, rules, format instructions |
| `HumanMessage` | user | End-user input |
| `AIMessage` | assistant | Model output; may include `tool_calls` |
| `ToolMessage` | tool | Result returned after executing a tool |

```
[SystemMessage]  You are a helpful assistant.
[HumanMessage]   What is 2 + 2?
[AIMessage]      4
[HumanMessage]   What about 3 + 3?
[AIMessage]      6
```

Prompt templates produce these message lists; chat models consume them. **03. Chat Models and Messages** covers invocation, streaming, and provider switching in detail.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

messages = [
    SystemMessage(content="Reply in one short sentence."),
    HumanMessage(content="What does a REST API do?"),
    AIMessage(content="A REST API exposes resources over HTTP using standard methods."),
    HumanMessage(content="Name one HTTP method."),
]

for i, m in enumerate(messages):
    print(f"{i+1}. [{m.type:10s}] {m.content}")

---

## 11. Your First Real LangChain Chain

Below is a minimal but complete chain: template → chat model → string parser. Replace `OPENAI_API_KEY` with a valid key to run live.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0.2,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant for API documentation questions."),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

answer = chain.invoke({"question": "In one sentence, what is OAuth2?"})
print(answer)

### 11.1 What Happened Step by Step

1. **`prompt.invoke({"question": ...})`** — returns a `ChatPromptValue` with formatted messages.
2. **`llm.invoke(messages)`** — calls OpenAI; returns an `AIMessage`.
3. **`StrOutputParser().invoke(ai_message)`** — extracts `.content` as a plain `str`.

Without `StrOutputParser`, `chain.invoke()` returns an `AIMessage` object. Parsers normalize output types so downstream code (API responses, JSON serialization) stays simple.

---

## 12. LangChain vs Raw Provider SDK

| Aspect | Raw OpenAI SDK | LangChain |
|--------|----------------|----------|
| **Learning curve** | Lower for one call | Higher initially |
| **Composition** | Manual functions | LCEL pipe |
| **Prompt reuse** | f-strings everywhere | `ChatPromptTemplate` |
| **Provider swap** | Rewrite client code | Change model class / `init_chat_model` |
| **Streaming** | Manual chunk loop | `chain.stream()` |
| **Tracing** | Custom logging | Callbacks / LangSmith |
| **Dependencies** | Minimal | Several packages |

**When to stay with raw SDK:** a single endpoint, one prompt, no retrieval, no tools — a 20-line function is fine.

**When LangChain pays off:** three or more steps, multiple prompts, RAG, tools, memory, or you need streaming/tracing consistently across routes.

---

## 13. Common Application Patterns

LangChain supports the dominant GenAI architectures:

| Pattern | LangChain pieces | Notebook |
|---------|------------------|----------|
| **Q&A chatbot** | prompt, llm, parser | **03–05** |
| **Document Q&A (RAG)** | retriever, prompt, llm | **08–11** |
| **Structured extraction** | parser / `with_structured_output` | **05** |
| **Tool-using assistant** | `@tool`, `bind_tools`, `create_agent` | **12–13** |
| **Multi-turn chat** | `RunnableWithMessageHistory` | **14** |
| **Classification router** | `RunnableBranch` | **06** |

```
RAG (preview — full build in 11):

question ──► retriever ──► format_docs ──► prompt ──► llm ──► answer
```

---

## 14. The LangChain Ecosystem

LangChain the **library** is part of a broader toolchain maintained by LangChain Inc.:

| Tool | Purpose |
|------|--------|
| **LangSmith** | Trace, evaluate, and monitor LLM apps ([smith.langchain.com](https://smith.langchain.com)) |
| **LangGraph** | Explicit state graphs for cyclic agent workflows (separate track in this curriculum) |
| **LangServe** | Deploy LangChain runnables as REST endpoints (legacy; many teams use FastAPI directly) |

You can use LangChain **without** LangSmith — but enabling tracing (environment variables + callbacks) dramatically speeds up debugging multi-step chains. **15. Callbacks, Caching, and Observability** covers this hands-on.

---

## 15. Installation Recap

From the repo root with the project virtual environment activated:

```bash
pip install langchain langchain-core langchain-openai langchain-chroma langchain-text-splitters chromadb
```

Optional for extended integrations:

```bash
pip install langchain-community
```

Environment variable for OpenAI:

```bash
export OPENAI_API_KEY=sk-...
```

Or set `OPENAI_API_KEY` in the notebook setup cell as shown above.

---

## 16. Design Principles to Carry Forward

As you progress through **02–17**, these principles recur:

1. **Compose, don't monolith** — small runnables piped together beat one giant function.
2. **Parse at the boundary** — use output parsers so API layers receive clean types.
3. **Keep provider logic in runnables** — your FastAPI route should call `chain.invoke`, not `openai_client` directly.
4. **Trace early** — add callbacks before production, not after the first outage.
5. **Pin versions** — GenAI libraries change quickly; lock dependencies in deployment.
6. **Wrap frameworks** — expose your own service class so LangChain can be replaced without rewriting the API.

In [ ]:
# Visualize how many building blocks a typical app uses

apps = ["Chatbot", "RAG Q&A", "Tool agent", "Production API"]
blocks_used = [3, 7, 9, 12]  # illustrative runnable count

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(apps, blocks_used, color=["#4c72b0", "#55a868", "#c44e52", "#8172b3"])
ax.set_xlabel("LangChain building blocks (illustrative)")
ax.set_title("Complexity grows with pattern — LangChain scales with you")
for bar, n in zip(bars, blocks_used):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2, str(n), va="center")
plt.tight_layout()
plt.show()

---

## 17. Common Beginner Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Following pre-2024 tutorials | Import errors, deprecated APIs | Use this 1.x track |
| Skipping `StrOutputParser` | Chain returns `AIMessage`, not `str` | Always terminate with a parser for text |
| Importing everything from `langchain` | Unclear dependencies | Import from `langchain_core`, provider packages |
| Putting business logic in prompts | Hard to test | Keep prompts as templates; logic in runnables |
| Adopting LangChain for one LLM call | Unnecessary complexity | Raw SDK is fine for trivial cases |
| No API key guard in notebooks | Cryptic auth errors | Set `OPENAI_API_KEY` before model cells |

---

## 18. Learning Path — This Series

```
01 Introduction (this notebook)
     │
     ▼
02 Runnable + LCEL ──► 03 Models ──► 04 Prompts ──► 05 Parsers
     │
     ▼
06 Composition ──► 07 Streaming/async
     │
     ▼
08 Documents ──► 09 Embeddings/vector store ──► 10 Retrievers ──► 11 RAG
     │
     ▼
12 Tools ──► 13 Agents ──► 14 Memory
     │
     ▼
15 Observability ──► 16 Fallbacks/config ──► 17 Production
```

Each notebook is **self-contained** — you can jump to **11. RAG with LCEL** if you already know runnables and prompts — but following the sequence builds concepts in order.

---

## 19. Summary

**LangChain** is a composable framework for LLM applications built on **`Runnable`** components and **LCEL** piping. The ecosystem splits across **`langchain-core`** (primitives), provider packages (`langchain-openai`, etc.), and **`langchain`** (agents and helpers).

Key takeaways from this introduction:

- LangChain solves **orchestration glue**, not modeling — you still choose models and write product logic.
- **LCEL** (`prompt | llm | parser`) is the modern replacement for legacy chain classes.
- Building blocks map to dedicated notebooks **02–17** in this track.
- Start simple (one chain), add retrieval, tools, memory, and observability as requirements grow.

Demonstrations verified package versions, composed Runnables without an API, inspected message types, and ran a first live chain with `ChatOpenAI`.

Next: **02. Runnable Protocol and LCEL** — the full execution API, composition helpers, and the foundation every later notebook builds on.